In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Convert a Shapefile of point locations into multiple Parquet files.

Each point in the input Shapefile has a `top_layer` attribute (e.g., 
S2A_MSIL2A_20180928T090731_N0500_R050_T35TMN). The script groups all 
points having the same `top_layer` value and writes one Parquet file 
for each tile name.

Output structure (per file):
    FILENAME   LONGITUDE   LATITUDE

Example:
    S2A_MSIL2A_20180928T090731_N0500_R050_T35TMN.parquet

Usage:
    1. Place your input Shapefile (e.g., patch_centers.shp) in the same folder.
    2. Adjust `IN_SHP` and `OUT_DIR` paths in the CONFIG section below.
    3. Run:  python3 shp_to_parquet.py
"""

# --------------------------------------------------------------------
# Imports
# --------------------------------------------------------------------
import os, re
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

# --------------------------------------------------------------------
# CONFIGURATION
# --------------------------------------------------------------------
IN_SHP   = "patch_centers.shp"         # Input Shapefile (points with top_layer field)
OUT_DIR  = "folder_for_parquet_files"  # Output folder for Parquet files
ENGINE   = "pyarrow"                   # Parquet engine: "pyarrow" or "fastparquet"
COMPR    = None                        # Compression: "snappy", "gzip", or None
TARGET_CRS = "EPSG:4326"               # Target coordinate reference system
OVERWRITE = True                       # If False, skip existing files
# --------------------------------------------------------------------

def sanitize(name: str) -> str:
    """
    Replace unsafe characters in filenames with underscores.
    Ensures compatibility across file systems.
    """
    return re.sub(r'[^A-Za-z0-9._\-]+', '_', str(name))

def col_ci(df, wanted):
    """
    Case-insensitive column lookup.
    Returns the actual column name that matches `wanted`.
    Raises KeyError if not found.
    """
    wl = wanted.lower()
    for c in df.columns:
        if c.lower() == wl:
            return c
    raise KeyError(f"Column '{wanted}' not found (case-insensitive).")

def main():
    # --------------------------------------------------------------
    # 1. Prepare environment
    # --------------------------------------------------------------
    os.makedirs(OUT_DIR, exist_ok=True)         # Create output folder if missing
    gdf = gpd.read_file(IN_SHP)                 # Read shapefile into GeoDataFrame

    if gdf.empty:
        raise RuntimeError("Input shapefile has no features.")

    # --------------------------------------------------------------
    # 2. Verify presence of 'top_layer' and CRS
    # --------------------------------------------------------------
    tl = col_ci(gdf, "top_layer")               # Find column, ignoring case

    if gdf.crs is None:
        raise ValueError("Input shapefile has no CRS; please define it before running.")

    # Reproject to WGS84 (lon/lat) if needed
    if str(gdf.crs) != TARGET_CRS:
        gdf = gdf.to_crs(TARGET_CRS)

    # --------------------------------------------------------------
    # 3. Handle MultiPoint geometries (if present)
    # --------------------------------------------------------------
    if (gdf.geometry.geom_type == "MultiPoint").any():
        gdf = gdf.explode(index_parts=False).reset_index(drop=True)

    # --------------------------------------------------------------
    # 4. Extract longitude / latitude from point geometry
    # --------------------------------------------------------------
    gdf["LONGITUDE"] = gdf.geometry.x
    gdf["LATITUDE"]  = gdf.geometry.y

    # Remove invalid or missing coordinates
    gdf = gdf[
        pd.to_numeric(gdf["LONGITUDE"], errors="coerce").notna() &
        pd.to_numeric(gdf["LATITUDE"],  errors="coerce").notna()
    ].copy()

    # --------------------------------------------------------------
    # 5. Group points by tile name (top_layer)
    # --------------------------------------------------------------
    tiles = gdf[tl].astype(str)
    groups = gdf.groupby(tiles, dropna=True)

    total_points = 0

    # --------------------------------------------------------------
    # 6. Loop through each group and write to Parquet
    # --------------------------------------------------------------
    for tile, sub in tqdm(groups, desc="Writing Parquet per tile"):
        safe = sanitize(tile)
        out_pq = os.path.join(OUT_DIR, f"{safe}.parquet")

        # Skip if file exists and overwrite disabled
        if (not OVERWRITE) and os.path.exists(out_pq):
            continue

        # Create the dataframe with desired structure
        df_out = pd.DataFrame({
            "FILENAME":  [tile] * len(sub),
            "LONGITUDE": sub["LONGITUDE"].to_numpy(),
            "LATITUDE":  sub["LATITUDE"].to_numpy(),
        })

        # Write parquet file with optional compression
        df_out.to_parquet(
            out_pq, 
            index=False, 
            engine=ENGINE,
            compression=COMPR if ENGINE == "pyarrow" else None
        )

        total_points += len(df_out)

    # --------------------------------------------------------------
    # 7. Final summary
    # --------------------------------------------------------------
    print(f"✅ Done. Tiles written: {groups.ngroups}, total points: {total_points}")
    print(f"📁 Output folder: {OUT_DIR}")

# --------------------------------------------------------------------
# Run main function
# --------------------------------------------------------------------
if __name__ == "__main__":
    main()
